***

# **Congestion Data Formatting Script**

***


This file contains a script for formatting Annual Hours of Peak Hour Excessive Delay Per Capita for use of RITIS data. The prerequisites needed for this to work, is to manually download all of the PHED files for the given UZA's, and to store all of these files into one folder. In addition, all of these files should be named using the following example schematic: Annual Hours PHED Per Capita_3-7pm_Austin_TX. The only thing the user will be changing in the file name should be the city and the state abbreviation. The PHED files can be made via the [NPMRDS analytics tool](https://npmrds.ritis.org/analytics/my-dashboard/) and selecting the MAP-21 widget. Search for your UZA, select the Annual Hours of Peak Hour Excessive Delay Per Capita box, and then add all of your years. You will be redirected to see a chart for all of the years selected detailing PHED. You can save this data, in the top right of the panel widget. A good tip to know, is you are able to simply edit your already existing widget to swap out the UZA and the name of the file, rather than re-inputting all of the years again. Additionally, to get the Percent of Eligible Miles missing PHED, you need to make each UZA's dashboard to only feature the last year with full data. So for 2024, we use 2023 data. From there, you look at the bottom right of the chart produced and subtract that number from 100%. 

***

## **Congestion_1**

***

In [72]:
import os
import pandas as pd

In [73]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths

# Git
path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')

# AGOL Path for Pete
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Congestion Data')

# Path with all of the PHED files named via the aforementioned schematic
path_congestion = os.path.join(path_sp, 'Data', 'Safe Equitable Resilient Infrastructure', 'Congestion')
path_phed = os.path.join(path_congestion, 'RTIS', 'PHED')
path_lottr = os.path.join(path_congestion, 'RTIS', 'LOTTR')

print(user)
print(path_git)

jchoy
C:\Users\jchoy\Documents\Projects\Regional-Monitoring\Indicator_Gen


<>:11: SyntaxWarning: invalid escape sequence '\R'
<>:11: SyntaxWarning: invalid escape sequence '\R'
C:\Users\jchoy\AppData\Local\Temp\ipykernel_34484\2183128991.py:11: SyntaxWarning: invalid escape sequence '\R'
  path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')


In [74]:
# Importing the data to follow a naming pattern

df_list = []

# Iterate over every file in path
for filename in os.listdir(path_phed):
    if filename.endswith('.csv'):
        
        # Extract city name from the filename
        uza = filename.split('_')[2]
        
        # Load the file into a df
        file_path = os.path.join(path_phed, filename)
        df = pd.read_csv(file_path)
        
        # Adding new col w/ UZA name. This is for joining later
        df['UZA'] = uza

        df_list.append(df)

In [75]:
# Joining now

congestion_1 = pd.concat(df_list, axis = 0, ignore_index=True)
congestion_1['Month'] = pd.to_datetime(congestion_1['Month']) #format = "%Y/%m"
congestion_1['Month'] = congestion_1['Month'].dt.to_period('M')

congestion_1_wide = congestion_1.pivot_table(index='Month', columns='UZA', values='PHED (hours)', aggfunc='mean', dropna=False)
congestion_1_wide.reset_index()

C:\Users\jchoy\AppData\Local\Temp\ipykernel_34484\4044278035.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  congestion_1['Month'] = pd.to_datetime(congestion_1['Month']) #format = "%Y/%m"


UZA,Month,Austin,Charlotte,Cincinatti,Cleveland,Columbus,Detroit,Indianapolis,Kansas City,Miami,...,Pittsburgh,Portland,Riverside-San Bernadino,Sacramento,San Antonio,San Diego,San Francisco-Oakland,San Jose,St Louis,Tampa-St Petersburg
0,2011-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2011-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2011-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2011-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2011-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,2024-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
164,2024-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
165,2024-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
166,2024-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [76]:
# Get yearly averages for each UZA

congestion_1_year = congestion_1.copy()
congestion_1_year['Year'] = congestion_1_year['Month'].dt.year

# Group by 'Year' and 'UZA', then calculate the mean

congestion_1_year = congestion_1_year.groupby(['Year', 'UZA']).mean().reset_index()
congestion_1_year = congestion_1_year[['Year', 'UZA', 'PHED (hours)']]

# Now we need to do this for the wide format

congestion_1_wide_year = congestion_1_year.pivot_table(index='Year', columns='UZA', values='PHED (hours)', aggfunc='mean', dropna=False)
congestion_1_wide_year.reset_index()

UZA,Year,Austin,Charlotte,Cincinatti,Cleveland,Columbus,Detroit,Indianapolis,Kansas City,Miami,...,Pittsburgh,Portland,Riverside-San Bernadino,Sacramento,San Antonio,San Diego,San Francisco-Oakland,San Jose,St Louis,Tampa-St Petersburg
0,2011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2012,0.783333,0.291667,0.308333,0.158333,0.241667,0.233333,NaN,0.100000,0.100000,...,0.150000,0.000000,0.250000,0.191667,0.450000,0.391667,0.708333,0.358333,0.250000,0.108333
2,2013,1.316667,0.708333,0.408333,0.258333,0.291667,0.683333,NaN,0.166667,0.350000,...,0.441667,0.058333,0.600000,0.433333,0.733333,0.675000,1.225000,0.983333,0.425000,0.308333
3,2014,2.066667,1.100000,0.725000,0.733333,0.616667,1.641667,NaN,0.250000,0.758333,...,0.850000,0.133333,0.891667,0.800000,1.141667,1.050000,1.833333,1.800000,0.658333,0.566667
4,2015,2.150000,1.100000,0.816667,0.850000,0.766667,1.558333,NaN,0.375000,0.808333,...,0.941667,0.175000,1.033333,0.933333,1.241667,1.208333,2.000000,2.025000,0.750000,0.575000
5,2016,1.950000,1.275000,0.850000,0.658333,0.841667,1.058333,NaN,0.283333,1.000000,...,0.800000,1.333333,1.291667,1.216667,0.925000,1.491667,2.541667,2.225000,0.816667,0.708333
6,2017,1.816667,1.275000,0.908333,0.641667,0.908333,1.208333,NaN,0.300000,1.033333,...,0.833333,1.366667,1.366667,1.233333,0.866667,1.550000,2.541667,2.225000,0.766667,0.741667
7,2018,1.866667,1.450000,0.950000,0.675000,1.083333,1.258333,NaN,0.350000,1.025000,...,1.000000,1.341667,1.341667,1.416667,0.966667,1.616667,2.608333,2.400000,0.800000,0.808333
8,2019,1.858333,1.233333,0.733333,0.516667,0.608333,0.975000,NaN,0.316667,0.850000,...,0.841667,1.125000,1.358333,1.266667,0.983333,1.408333,2.558333,2.283333,0.800000,0.725000
9,2020,0.900000,0.633333,0.416667,0.291667,0.258333,0.533333,NaN,0.166667,0.475000,...,0.450000,0.500000,0.816667,0.575000,0.508333,0.608333,1.041667,0.966667,0.416667,0.408333


***

## **Congestion_3**

***

For Congestion_3, we only need data for SACOG counties. To do this, we use the same tool as in Congestion_1, but now we use the MPA for SACOG instead. Select the first three measures, and do the same as we did before. 

In [77]:
# Loading the files

truck_path = os.path.join(path_lottr, 'Truck Travel Time Reliability - Sacramento.csv')
interstate_path = os.path.join(path_lottr, 'Interstate Travel Time Reliability - Sacramento.csv')
nonint_path = os.path.join(path_lottr, 'Non-interstate NHS Travel Time Reliability - Sacramento.csv')

# reading files

truck_travel = pd.read_csv(truck_path)
interstate_travel = pd.read_csv(interstate_path)
nonint_travel = pd.read_csv(nonint_path)

# specifying LOTTR for int and non-int

interstate_travel.rename(columns={'LOTTR (%)': 'Interstate LOTTR (%)'}, inplace=True)
nonint_travel.rename(columns={'LOTTR (%)': 'Non-Interstate LOTTR (%)'}, inplace=True)

In [78]:
# Merge

congestion_3 = truck_travel.merge(interstate_travel, on='Month', how='left').merge(nonint_travel, on='Month', how='left')
congestion_3['Month'] = pd.to_datetime(congestion_3['Month']) #format = "%Y/%m"
congestion_3['Month'] = congestion_3['Month'].dt.to_period('M')

# Get the averages per year now

congestion_3_year = congestion_3.copy()
congestion_3_year['Year'] = congestion_3_year['Month'].dt.year

# Group by 'Year' and 'UZA', then calculate the mean

congestion_3_year = congestion_3_year.groupby('Year')[['TTTR (%)', 'Interstate LOTTR (%)', 'Non-Interstate LOTTR (%)']].mean().reset_index()
#congestion_3_year = congestion_3_year.drop(columns = 'Month')

C:\Users\jchoy\AppData\Local\Temp\ipykernel_34484\2316967675.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  congestion_3['Month'] = pd.to_datetime(congestion_3['Month']) #format = "%Y/%m"


***

## **Exports**

***

In [86]:
# two exports for data folder | one in date time other in weighted average per year

# one in task 8 for average per year

# we do these exports for both congestion_1 and congestion_3

# Exports
indicator_name = 'Congestion'

# congestion_1
one_output_xlsx = [indicator_name, '_', '1', '.xlsx']
one_output_xlsx = "".join(one_output_xlsx)
one_output_csv = [indicator_name, '_', '1', '.csv']
one_output_csv = "".join(one_output_csv)

# congestion_3
three_output_xlsx = [indicator_name, '_', '3', '.xlsx']
three_output_xlsx = "".join(three_output_xlsx)
three_output_csv = [indicator_name, '_', '3', '.csv']
three_output_csv = "".join(three_output_csv)

In [89]:
# Set file path for exporting

path_out_one = os.path.join(path_congestion, 'Congestion_1')
path_out_three = os.path.join(path_congestion, 'Congestion_3')

# Congestion_1 Exports to SP
with pd.ExcelWriter(os.path.join(path_out_one, one_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    congestion_1.to_excel(writer, index = False, sheet_name = 'UZA Long'       )
    congestion_1_wide.to_excel(writer, index = False, sheet_name = 'UZA Wide'  )

# Congestion_1 Exports AGOL
congestion_1_year.to_csv(os.path.join(path_agol, one_output_csv), index = False)
congestion_1_wide_year.to_csv(os.path.join(path_agol, 'Congestion_1 Wide.csv'))

# Congestion_3 Exports
with pd.ExcelWriter(os.path.join(path_out_three, three_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    congestion_3.to_excel(writer, index = False, sheet_name = 'SACOG'       )

congestion_3_year.to_csv(os.path.join(path_agol, three_output_csv), index = False)

print('Congestion_1 SP Files Exported Here: ' + path_out_one)
print('Congestion_3 Files Exported Here: '   + path_out_three)
print('Congestion_1 & Congestion_3 AGOL Files Exported Here: ' + path_agol)

Congestion_1 SP Files Exported Here: C:\Users\jchoy\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Data\Safe Equitable Resilient Infrastructure\Congestion\Congestion_1
Congestion_3 Files Exported Here: C:\Users\jchoy\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Data\Safe Equitable Resilient Infrastructure\Congestion\Congestion_3
Congestion_1 & Congestion_3 AGOL Files Exported Here: C:\Users\jchoy\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Process Revamp\Task 8. Reproduce Progress Report indicators\Indicator Data\Congestion Data
